# 07 — Gameweek Review: Predicted vs Actual

Compares the model's predictions (made from GW30 data) against actual GW31 results.

**Flow:**
1. Load GW30 feature snapshot and generate predictions for GW31
2. Run the optimizer to select the optimal starting XI
3. Fetch actual GW31 points from the FPL API
4. Compare predicted vs actual for the XI and the full player pool

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from src.models.predict import FPLPredictor
from src.models.optimize import FPLOptimizer

PROCESSED = Path('../data/processed')
RAW       = Path('../data/raw')
MODELS    = Path('../models')

POSITION_MAP = {1: 'GKP', 2: 'DEF', 3: 'MID', 4: 'FWD'}
print('Libraries loaded.')

## 1. Generate predictions from GW30 snapshot

In [ ]:
features = pd.read_parquet(PROCESSED / 'features.parquet')

# GW30 is the last complete snapshot — model predicts GW31 from this
PREDICT_FROM_GW = 30
ACTUAL_GW       = 31

gw30 = features[features['round'] == PREDICT_FROM_GW].copy()
print(f'GW{PREDICT_FROM_GW} snapshot: {len(gw30)} players')

predictor = FPLPredictor(models_dir=MODELS)
predictor.load()

gw30['predicted_pts'] = predictor.predict(gw30)
print(f'Predictions generated. Range: {gw30["predicted_pts"].min():.2f} — {gw30["predicted_pts"].max():.2f}')

In [ ]:
# Build player pool (same as optimizer notebook)
fpl_players = pd.read_parquet(RAW / 'fpl_players.parquet')
fpl_players['position'] = fpl_players['element_type'].map(POSITION_MAP)
fpl_meta = fpl_players[['id', 'web_name', 'position', 'team', 'status', 'team_code']].rename(columns={'id': 'player_id'})

# Load team names
fpl_teams = pd.read_parquet(RAW / 'fpl_teams.parquet')[['id', 'short_name']].rename(
    columns={'id': 'team', 'short_name': 'team_name'}
)

pool = gw30[['player_id', 'value', 'predicted_pts']].copy()
pool.rename(columns={'value': 'now_cost'}, inplace=True)
pool = pool.merge(fpl_meta, on='player_id', how='left')
pool = pool.merge(fpl_teams, on='team', how='left')
pool = pool.dropna(subset=['now_cost', 'position', 'predicted_pts'])
pool = pool[pool['now_cost'] > 0]

players_available = pool[pool['status'] == 'a'].copy()
print(f'Available player pool: {len(players_available)} players')

## 2. Select optimal starting XI (as if picking before GW31)

In [ ]:
optimizer = FPLOptimizer(budget=100.0)
xi_result = optimizer.select_starting_xi(players_available, budget=100.0)

print(f'Status:          {xi_result["status"]}')
print(f'Formation:       {xi_result["formation"]}')
print(f'Captain:         {xi_result["captain"]}')
print(f'Predicted total: {xi_result["predicted_total"]:.2f} pts (with captain double)')

xi_df = xi_result['xi'].copy()
print('\n=== Predicted Starting XI ===')
for pos in ['GKP', 'DEF', 'MID', 'FWD']:
    for _, row in xi_df[xi_df['position'] == pos].iterrows():
        cap = ' (C)' if row['web_name'] == xi_result['captain'] else ''
        print(f'  {row["web_name"]:<22} {pos}  {row["predicted_pts"]:.2f} pts pred{cap}')

## 3. Fetch actual GW31 points from FPL API

In [ ]:
BASE_URL = 'https://fantasy.premierleague.com/api/'

def fetch_gw_points(player_id: int, gw: int) -> float | None:
    """Fetch actual points for a player in a specific gameweek."""
    url = f'{BASE_URL}element-summary/{player_id}/'
    r = requests.get(url, timeout=10)
    if r.status_code != 200:
        return None
    history = r.json().get('history', [])
    for h in history:
        if h.get('round') == gw:
            return h.get('total_points')
    return None  # no fixture that GW (blank)

print(f'Fetching GW{ACTUAL_GW} actual points for XI...')
actuals = {}
for _, row in xi_df.iterrows():
    pts = fetch_gw_points(int(row['player_id']), ACTUAL_GW)
    actuals[row['web_name']] = pts
    status = f'{pts} pts' if pts is not None else 'blank GW'
    print(f'  {row["web_name"]:<22} → {status}')

xi_df['actual_pts'] = xi_df['web_name'].map(actuals)
xi_df['blank_gw']   = xi_df['actual_pts'].isna()
xi_df['actual_pts'] = xi_df['actual_pts'].fillna(0)

## 4. Results: predicted vs actual

In [ ]:
# Apply captain doubling to actuals too
cap_name = xi_result['captain']
xi_df['predicted_pts_with_cap'] = xi_df.apply(
    lambda r: r['predicted_pts'] * 2 if r['web_name'] == cap_name else r['predicted_pts'], axis=1
)
xi_df['actual_pts_with_cap'] = xi_df.apply(
    lambda r: r['actual_pts'] * 2 if r['web_name'] == cap_name else r['actual_pts'], axis=1
)
xi_df['error'] = xi_df['actual_pts'] - xi_df['predicted_pts']

total_pred   = xi_df['predicted_pts_with_cap'].sum()
total_actual = xi_df['actual_pts_with_cap'].sum()
mae          = xi_df['error'].abs().mean()

print('=' * 60)
print(f'  Predicted total (with captain): {total_pred:.2f} pts')
print(f'  Actual total   (with captain): {total_actual:.0f} pts')
print(f'  Difference:                    {total_actual - total_pred:+.1f} pts')
print(f'  MAE (per player, no cap):      {mae:.2f} pts')
print('=' * 60)
print()

# Per-player breakdown
print(f'{"Player":<22} {"Pos":<5} {"Pred":>6} {"Actual":>7} {"Error":>7}')
print('-' * 50)
for _, row in xi_df.sort_values('position').iterrows():
    cap   = '(C)' if row['web_name'] == cap_name else '   '
    blank = ' [blank]' if row['blank_gw'] else ''
    print(f'{row["web_name"]:<22} {row["position"]:<5} {row["predicted_pts"]:>6.2f} '
          f'{int(row["actual_pts"]):>7} {row["error"]:>+7.2f} {cap}{blank}')

## 5. Visualise predicted vs actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1a1a2e')

# Sort by predicted pts
plot_df = xi_df.sort_values('predicted_pts', ascending=True)
names   = plot_df['web_name'].tolist()
pred    = plot_df['predicted_pts'].tolist()
actual  = plot_df['actual_pts'].tolist()
errors  = plot_df['error'].tolist()
bar_colours = ['#e74c3c' if e < 0 else '#00bc8c' for e in errors]

# ── Left: side-by-side bar chart ─────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#16213e')
y   = np.arange(len(names))
h   = 0.35

ax.barh(y + h/2, pred,   height=h, label='Predicted', color='#375a7f', alpha=0.9)
ax.barh(y - h/2, actual, height=h, label='Actual',    color='#00bc8c', alpha=0.9)

ax.set_yticks(y)
ax.set_yticklabels(names, color='white', fontsize=9)
ax.set_xlabel('Points', color='white')
ax.set_title(f'GW{ACTUAL_GW}: Predicted vs Actual', color='white', fontweight='bold')
ax.tick_params(colors='white')
ax.legend(labelcolor='white', framealpha=0.3)
for spine in ax.spines.values():
    spine.set_edgecolor('#444')

# Mark captain
cap_idx = names.index(cap_name) if cap_name in names else None
if cap_idx is not None:
    ax.annotate('(C)', xy=(max(pred[cap_idx], actual[cap_idx]) + 0.1, cap_idx),
                color='#f5a623', fontsize=9, va='center')

# ── Right: error bar chart ────────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#16213e')

ax2.barh(y, errors, color=bar_colours, alpha=0.9, height=0.6)
ax2.axvline(x=0, color='white', linewidth=1.2)

ax2.set_yticks(y)
ax2.set_yticklabels(names, color='white', fontsize=9)
ax2.set_xlabel('Actual − Predicted (pts)', color='white')
ax2.set_title('Prediction error per player', color='white', fontweight='bold')
ax2.tick_params(colors='white')
for spine in ax2.spines.values():
    spine.set_edgecolor('#444')

legend_elements = [
    mpatches.Patch(color='#00bc8c', label='Underestimated (model too low)'),
    mpatches.Patch(color='#e74c3c', label='Overestimated (model too high)'),
]
ax2.legend(handles=legend_elements, labelcolor='white', framealpha=0.3, fontsize=8)

plt.suptitle(
    f'GW{ACTUAL_GW} Review  |  Pred: {total_pred:.1f}pts  |  Actual: {total_actual:.0f}pts  |  MAE: {mae:.2f}',
    color='white', fontsize=11, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig(MODELS / f'gw{ACTUAL_GW}_review.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(f'Saved to models/gw{ACTUAL_GW}_review.png')

## 6. Top predicted players vs what they actually scored

How well did the model rank the top players across the full pool (not just the XI)?

In [ ]:
print(f'Fetching GW{ACTUAL_GW} actuals for top 30 predicted players...')
top30 = pool.sort_values('predicted_pts', ascending=False).head(30).copy()

top30_actuals = {}
for _, row in top30.iterrows():
    pts = fetch_gw_points(int(row['player_id']), ACTUAL_GW)
    top30_actuals[row['player_id']] = pts if pts is not None else 0

top30['actual_pts']  = top30['player_id'].map(top30_actuals)
top30['pred_rank']   = range(1, len(top30) + 1)
top30['actual_rank'] = top30['actual_pts'].rank(ascending=False, method='min').astype(int)
top30['error']       = top30['actual_pts'] - top30['predicted_pts']

print(f'\n{"Rank":<5} {"Player":<22} {"Pos":<5} {"Pred":>6} {"Actual":>7} {"Error":>7}')
print('-' * 55)
for _, row in top30.iterrows():
    print(f'{int(row["pred_rank"]):<5} {row["web_name"]:<22} {row["position"]:<5} '
          f'{row["predicted_pts"]:>6.2f} {int(row["actual_pts"]):>7} {row["error"]:>+7.2f}')

In [ ]:
# Scatter: predicted vs actual for top 30
fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#16213e')

pos_colours = {'GKP': '#f5a623', 'DEF': '#375a7f', 'MID': '#00bc8c', 'FWD': '#e74c3c'}

for pos, colour in pos_colours.items():
    sub = top30[top30['position'] == pos]
    ax.scatter(sub['predicted_pts'], sub['actual_pts'],
               c=colour, label=pos, s=80, zorder=5, alpha=0.9)

# Perfect prediction line
lim = max(top30['predicted_pts'].max(), top30['actual_pts'].max()) + 1
ax.plot([0, lim], [0, lim], '--', color='white', linewidth=1, alpha=0.5, label='Perfect prediction')

# Label each point
for _, row in top30.iterrows():
    ax.annotate(row['web_name'], (row['predicted_pts'], row['actual_pts']),
                fontsize=6.5, color='#ddd', xytext=(4, 2), textcoords='offset points')

ax.set_xlabel('Predicted pts', color='white')
ax.set_ylabel('Actual pts', color='white')
ax.set_title(f'Top 30 predicted players — GW{ACTUAL_GW}', color='white', fontweight='bold')
ax.tick_params(colors='white')
ax.legend(labelcolor='white', framealpha=0.3)
for spine in ax.spines.values():
    spine.set_edgecolor('#444')

# Correlation
corr = top30[['predicted_pts', 'actual_pts']].corr().iloc[0, 1]
ax.text(0.05, 0.93, f'Pearson r = {corr:.2f}',
        transform=ax.transAxes, color='white', fontsize=9,
        bbox={'boxstyle': 'round', 'facecolor': '#333', 'alpha': 0.7})

plt.tight_layout()
plt.savefig(MODELS / f'gw{ACTUAL_GW}_scatter.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(f'Saved to models/gw{ACTUAL_GW}_scatter.png')

## 7. Summary

In [ ]:
overestimated  = (xi_df['error'] < -1).sum()
underestimated = (xi_df['error'] > 1).sum()
close          = ((xi_df['error'].abs()) <= 1).sum()

print('=== GW Review Summary ===')
print(f'  Predicted total (XI + cap): {total_pred:.1f} pts')
print(f'  Actual total   (XI + cap): {total_actual:.0f} pts')
print(f'  Gap:                       {total_actual - total_pred:+.1f} pts')
print(f'  MAE per player:            {mae:.2f} pts')
print()
print(f'  Players within 1pt:        {close}/11')
print(f'  Overestimated (>1pt high): {overestimated}/11')
print(f'  Underestimated (>1pt low): {underestimated}/11')
print()
best  = xi_df.loc[xi_df['error'].idxmax()]
worst = xi_df.loc[xi_df['error'].idxmin()]
print(f'  Biggest miss (over):  {worst["web_name"]} — pred {worst["predicted_pts"]:.1f}, actual {int(worst["actual_pts"])} ({worst["error"]:+.1f})')
print(f'  Best surprise (under): {best["web_name"]} — pred {best["predicted_pts"]:.1f}, actual {int(best["actual_pts"])} ({best["error"]:+.1f})')